# FloodRoad-SAM3 Colab Runner

This notebook runs the FloodRoad-SAM3 pipeline from GitHub. The first section is a fast smoke test on synthetic processed tiles; the second section contains the real SpaceNet 8 commands.

Experiment default: CC-RL uses 20 training tiles and evaluation reuses exactly those 20 tile ids via `runs/default/rl_samples.json`.

## 1. Clone and install

In [ ]:
!git clone https://github.com/ezh2020/FloodRoad-SAM3.git /content/FloodRoad-SAM3 || true
%cd /content/FloodRoad-SAM3
!git pull --ff-only
!pip -q install -r requirements.txt

## 2. Fast smoke test

This verifies imports, dataset loading, training loops, CC-RL sample selection, and table generation without downloading SpaceNet 8 or SAM3 weights. Do not report smoke-test metrics as experiment results.

In [ ]:
!python data/make_toy_dataset.py --processed-root /content/spacenet8/toy_processed --num-tiles 24 --tile-size 128

import yaml
from pathlib import Path

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['paths']['processed_root'] = '/content/spacenet8/toy_processed'
cfg['paths']['output_dir'] = '/content/floodroad_runs/toy_smoke'
cfg['sam3']['allow_mock'] = True
cfg['sam3']['device'] = 'auto'
cfg['deeplab']['pretrained_backbone'] = False
cfg['data']['num_workers'] = 0
cfg['eval']['warmup'] = 1
cfg['eval']['iterations'] = 2
Path('/content/floodroad_runs').mkdir(parents=True, exist_ok=True)
Path('/content/floodroad_runs/toy_smoke.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print(Path('/content/floodroad_runs/toy_smoke.yaml').read_text())

In [ ]:
!python train.py --config /content/floodroad_runs/toy_smoke.yaml --method deeplab --smoke
!python train.py --config /content/floodroad_runs/toy_smoke.yaml --method ours_no_tm --smoke
!python train.py --config /content/floodroad_runs/toy_smoke.yaml --method ours_tm --smoke
!python evaluate.py --config /content/floodroad_runs/toy_smoke.yaml --skip-efficiency
!python evaluate.py --config /content/floodroad_runs/toy_smoke.yaml --efficiency-only

## 3. Real SpaceNet 8 run

Upload or mount the raw SpaceNet 8 files under `/content/spacenet8/raw`, or provide a `pairs.csv` with `id,pre_path,post_path,road_geojson_path,flood_path`. Keep `sam3.allow_mock: false` for real results.

In [ ]:
# Optional full public bucket download. This can be large.
# !python data/download.py --output /content/spacenet8/raw

# Preprocess raw SpaceNet 8 scenes into 1024x1024 tiles.
# !python data/preprocess.py --config configs/default.yaml --raw-root /content/spacenet8/raw --processed-root /content/spacenet8/processed

# If you have an explicit pairs CSV, use this instead.
# !python data/preprocess.py --config configs/default.yaml --raw-root /content/spacenet8/raw --processed-root /content/spacenet8/processed --pairs-csv /content/spacenet8/pairs.csv

In [ ]:
# Full training. These are intentionally long-running.
# !python train.py --config configs/default.yaml --method deeplab
# !python train.py --config configs/default.yaml --method ours_no_tm
# !python train.py --config configs/default.yaml --method ours_tm

# Table 1: accuracy on the 20 CC-RL sample tiles.
# !python evaluate.py --config configs/default.yaml --skip-efficiency

# Table 2: efficiency for all four configurations.
# !python evaluate.py --config configs/default.yaml --efficiency-only

## Outputs

The generated tables are written to the configured run directory as `accuracy_table.md`, `accuracy_table.csv`, `efficiency_table.md`, and `efficiency_table.csv`. The 20 RL/evaluation tile ids are saved as `rl_samples.json`.